# Eksplorasi AutoModelForSequenceClassification

**Task**: Klasifikasi Teks / Sequence (Sentimen analisis, Deteksi topik, Spam detection)
**Cara Kerja**: Menerima satu urutan teks penuh (satu kalimat atau paragraf), memprosesnya, dan mengeluarkan 1 label kategori untuk keseluruhan teks tersebut.
**Model Populer**: BERT, RoBERTa, DistilBERT, varian klasifikasi lainnya.
**Dataset**: `imdb` - dataset ulasan film (Movie Reviews) yang sangat populer digunakan untuk melatih tugas analisis sentimen (positif/negatif).

In [ ]:
!pip install -q transformers datasets torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import torch

## 1. Load Dataset Publik (`imdb`)
Dataset IMDB berisi teks ulasan film dari pengguna dan memiliki label sentimen (0 untuk negatif, 1 untuk positif). Ini adalah representasi sempurna untuk task klasifikasi satu teks utuh (Sequence Classification).

In [ ]:
dataset = load_dataset("imdb", split="train")

# Menampilkan nama-nama label
features = dataset.features
print("Daftar Label:", features["label"].names)

print("\n--- Contoh Data Index-0 ---")
text_sample = dataset[0]["text"]
label_sample = dataset[0]["label"]
print(f"Teks: {text_sample[:250]}... (dipotong)")
print(f"Label ID: {label_sample} -> {features['label'].names[label_sample]}")

## 2. Load Tokenizer & Model
Kita akan memuat model `distilbert-base-uncased-finetuned-sst-2-english`. Model ini merupakan versi kompak dari BERT yang sudah di fine-tune secara spesifik untuk tugas klasifikasi sentimen.

In [ ]:
model_checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Menggunakan AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint)

## 3. Inferensi Manual
Berbeda dengan Token Classification yang memberikan logits untuk tiap bagian kata, Sequence Classification akan mengambil semua kata lalu mengeluarkan satu list logits (probabilitas) akhir yang merepresentasikan seluruh kalimat.

In [ ]:
text = "I really loved this movie. The acting was spectacular and the storyline was deeply moving!"

inputs = tokenizer(text, return_tensors="pt")

# Model mengembalikan 1 set logits untuk 1 input text
with torch.no_grad():
    outputs = model(**inputs)
    
logits = outputs.logits
predicted_class_id = logits.argmax().item()

print("--- Hasil Prediksi Manual ---")
print("Logits:", logits.numpy())
print("Predicted Class ID:", predicted_class_id)
print("Label:", model.config.id2label[predicted_class_id])

## 4. Persiapan Data untuk PyTorch Training Mandiri
Untuk task klasifikasi teks, kita cukup memetakan teks menjadi token `input_ids` dan memisahkan kolom label target dari imdb sebagai `labels`. Tensor ini nantinya akan diproses oleh fungsi CrossEntropy bawaan PyTorch secara langsung karena label berbentuk array 1 dimensi per baris.

In [ ]:
from torch.utils.data import Dataset, DataLoader

# Ambil sampel kecil untuk mempercepat simulasi (50 baris saja)
train_sample = dataset.select(range(50))

def preprocess_function(examples):
    # Padding dan pemotongan teks agar seragam untuk tensor PyTorch
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

tokenized_train = train_sample.map(preprocess_function, batched=True, remove_columns=["text"])

class SequenceClassificationDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"]),
            "attention_mask": torch.tensor(item["attention_mask"]),
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_dataloader = DataLoader(SequenceClassificationDataset(tokenized_train), batch_size=4, shuffle=True)
print(f"Total Batch Data Sequence Classification: {len(train_dataloader)}")
print("Selesai memecah Token Data Tensor!")

## 5. Proses PyTorch Training Loop
Proses training untuk klasifikasi sequence ini adalah arketipe *Fine-Tuning* standar yang paling sering dan mudah dilatih pada model BERT: `outputs.loss` akan dengan otomatis mengeluarkan hasil valid dari *Cross Entropy Classification Loss*.

In [ ]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 1
print("==== Memulai Training Sequence Classification ===")
model.train()
for epoch in range(epochs):
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Logits berupa [Batch Size, 2] (kategori positif dan negatif)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if step % 5 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f">> Rata-rata Train Loss Epoch {epoch+1}: {avg_train_loss:.4f}\n")

## 6. Evaluasi Kinerja (Akurasi Sentimen)
Untuk analisis sentimen atau klasifikasi kategori apapun secara umum, metrik mutlak yang paling umum digunakan adalah **Akrurasi** (proporsi klasifikasi prediksi yang dijawab secara benar terhadap target pelatihannya).

In [ ]:
val_sample = dataset.select(range(50, 70))
tokenized_val = val_sample.map(preprocess_function, batched=True, remove_columns=["text"])
val_dataloader = DataLoader(SequenceClassificationDataset(tokenized_val), batch_size=4)

model.eval()
total_val_loss = 0
correct_predictions = 0

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_val_loss += outputs.loss.item()
        
        predictions = outputs.logits.argmax(dim=-1)
        correct_predictions += (predictions == labels).sum().item()

avg_val_loss = total_val_loss / len(val_dataloader)
accuracy = correct_predictions / len(val_sample)

print("==== Hasil Evaluasi Klasifikasi ====")
print(f"Validation Loss : {avg_val_loss:.4f}")
print(f"Akurasi         : {accuracy:.4f} (atau {accuracy*100:.2f}%)")

## 7. Inference / Pengujian Model Secara Langsung
Mari kita uji sentimen sebuah teks untuk melihat apakah model mampu menebak kategori yang benar berdasarkan prediksinya.

In [ ]:
test_sentence = "I really loved the movie! The acting was fantastic and the plot was thrilling."

model.eval()
input_tensors = tokenizer(test_sentence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**input_tensors)
    
# Karena ini model dari id to label 1 untuk Sentimen Positif, maka Index tertinggi akan menampilkan prediksinya.
prediction_idx = outputs.logits.argmax(dim=-1).item()
confidence_score = torch.nn.functional.softmax(outputs.logits, dim=-1)[0][prediction_idx].item()

sentiment_label = "POSITIF" if prediction_idx == 1 else "NEGATIF"

print(f"Teks          : {test_sentence}")
print(f"Prediksi Sentimen : {sentiment_label} (Confidence: {confidence_score:.4f})")